# Rudder-step lateral response example

この Notebook では、`SampleGlider.stab` の線形空力モデルを使い、ラダーステップ入力に対する横転性能を次の順序で確認する。

1. 線形横・方向 4 状態モデルの有限時間応答を計算する。
2. 完全な状態行列から得られる Taylor 係数 $K_1$、$K_2$、$K_3$ を確認する。
3. 直接ロール、ラダー側力・上反角効果、ヨーレートロール、ヨーレートから横滑りを介する上反角効果の簡易経路係数を確認する。
4. 簡易ラダー横転指数

$$
K_{\mathrm{rudder,roll}}
=
\left(C_{l\hat r}-C_{l\beta}\right)C_{n\delta_r}
$$

を確認する。
5. 同じ `.stab`、質量、慣性モーメント、ラダー入力を用いて 6 自由度応答を計算する。
6. 線形モデルと 6 自由度モデルの有限時間ロール応答指標を比較する。
7. 6 自由度時刻歴を CSV と PNG へ保存する。

ここで使う質量・慣性モーメントは `SampleGlider` 用の数値例であり、G103A の実機 MassProp 値ではない。


## 1. import とリポジトリ位置


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "RollRudderGain.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/RollRudderGain.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.RollRudderGain import (  # noqa: E402
    calculate_linear_lateral_response_indices_from_stab,
    calculate_roll_response_index_by_delta_phi,
    plot_6dof_history,
    simulate_6dof_rudder_step_from_stab,
    write_6dof_history_csv,
)
from src.VSPAEROStab import read_vspaero_stab  # noqa: E402

print("repo_root:", repo_root)


## 2. 入力モデルと解析条件

`V`、`Bref`、`rho` は手入力せず、同じ `.stab` から読み取った値を線形モデル、6 自由度モデル、応答指数の無次元化に使用する。


In [ ]:
stab_path = (
    repo_root
    / "examples"
    / "models"
    / "SampleGlider"
    / "SampleGlider.stab"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "results"
    / "rudder_step_response"
)
output_dir.mkdir(parents=True, exist_ok=True)

if not stab_path.exists():
    raise FileNotFoundError(stab_path)

stab = read_vspaero_stab(stab_path)

mass = 100.0
inertia = {
    "Ixx": 1000.0,
    "Iyy": 75.0,
    "Izz": 1000.0,
    "Ixz": 0.0,
}
rho = stab.rho0
if rho is None:
    raise ValueError("SampleGlider.stab does not contain Rho.")

delta_r = math.radians(-10.0)
target_delta_phi = math.radians(2.0)
t_final = 5.0
max_step = 0.02

display(
    pd.Series(
        {
            "stab_path": str(stab_path),
            "mass": mass,
            "Ixx": inertia["Ixx"],
            "Iyy": inertia["Iyy"],
            "Izz": inertia["Izz"],
            "Ixz": inertia["Ixz"],
            "Vinf": stab.V0,
            "Bref": stab.Bref,
            "rho": rho,
            "delta_r_deg": math.degrees(delta_r),
            "target_delta_phi_deg": math.degrees(target_delta_phi),
            "t_final": t_final,
        },
        name="value",
    )
)


## 3. 線形横・方向モデル

状態量

$$
\mathbf{x}
=
\begin{bmatrix}
\beta & \hat p & \hat r & \phi
\end{bmatrix}^{\mathrm T}
$$

と無次元時間 $\tau=2Vt/b$ を使う線形モデルについて、指定した符号付きバンク角変化へ到達する時間と有限時間ロール応答指標を計算する。


In [ ]:
linear_response = calculate_linear_lateral_response_indices_from_stab(
    stab_path,
    mass=mass,
    Ixx=inertia["Ixx"],
    Izz=inertia["Izz"],
    Ixz=inertia["Ixz"],
    delta_r=delta_r,
    t_final=t_final,
    target_delta_phi=target_delta_phi,
    rho=rho,
    max_step=max_step,
)

display(
    pd.Series(
        {
            "reached": linear_response["linear_roll_response_reached"],
            "t_reach": linear_response["linear_t_reach"],
            "tau_reach": linear_response["linear_tau_reach"],
            "phi_final_deg": math.degrees(
                linear_response["linear_phi_final"]
            ),
            "finite_time_roll_index": linear_response[
                "linear_finite_time_roll_index"
            ],
            "error": linear_response["linear_roll_response_error"],
        },
        name="linear response",
    )
)


## 4. Taylor 係数と簡易経路係数

完全な線形モデルの Taylor 係数は、

$$
\frac{\Delta\phi}{\delta_r}
=
\frac{\tau^2}{2}K_1
+
\frac{\tau^3}{6}K_2
+
\frac{\tau^4}{24}K_3
+
\cdots
$$

に対応する。

簡易経路係数は、完全な $K_1,K_2,K_3$ をそのまま分解したものではなく、記事に示した近似条件の下で主要な物理経路を取り出した値である。


In [ ]:
taylor_and_paths = pd.Series(
    {
        "linear_taylor_K1": linear_response["linear_taylor_K1"],
        "linear_taylor_K2": linear_response["linear_taylor_K2"],
        "linear_taylor_K3": linear_response["linear_taylor_K3"],
        "simple_K1_direct_roll": linear_response[
            "simple_K1_direct_roll"
        ],
        "simple_K2_sideforce_dihedral": linear_response[
            "simple_K2_sideforce_dihedral"
        ],
        "simple_K2_yawrate_roll": linear_response[
            "simple_K2_yawrate_roll"
        ],
        "simple_K3_yawrate_to_beta_dihedral": linear_response[
            "simple_K3_yawrate_to_beta_dihedral"
        ],
        "simple_K3_reduced": linear_response["simple_K3_reduced"],
    },
    name="coefficient",
)

display(taylor_and_paths)
print("simple assumptions:")
print(linear_response["simple_taylor_assumptions"])


## 5. 簡易ラダー横転指数

記事で定義した簡易ラダー横転指数

$$
K_{\mathrm{rudder,roll}}
=
\left(C_{l\hat r}-C_{l\beta}\right)C_{n\delta_r}
$$

を安定微係数から直接計算する。`RollRudderGain.py` の `simple_turn_rate` は同じ量である。


In [ ]:
simple_rudder_roll_index = (
    linear_response["initial_Cl_rhat"]
    - linear_response["initial_Cl_beta"]
) * linear_response["initial_Cn_delta_r"]

if not math.isclose(
    simple_rudder_roll_index,
    linear_response["simple_turn_rate"],
    rel_tol=1.0e-12,
    abs_tol=1.0e-14,
):
    raise AssertionError(
        "The directly calculated simple rudder-roll index does not "
        "match simple_turn_rate."
    )

display(
    pd.Series(
        {
            "Cl_beta": linear_response["initial_Cl_beta"],
            "Cl_rhat": linear_response["initial_Cl_rhat"],
            "Cn_delta_r": linear_response["initial_Cn_delta_r"],
            "simple_rudder_roll_index": simple_rudder_roll_index,
            "simple_turn_rate": linear_response["simple_turn_rate"],
            "simple_turn_rate_full": linear_response[
                "simple_turn_rate_full"
            ],
        },
        name="value",
    )
)


## 6. 6 自由度ラダーステップ応答

この例では推力を使わないため `trim_thrust=False` とする。初期状態は完全な定常滑空トリムではない。

- 初期速度・迎角・横滑り角は `.stab` の基準状態から作る。
- 初期角速度は $p=q=r=0$ とする。
- `theta0=None` のため、初期ピッチ角は基準迎角に設定される。
- 推力はゼロである。

したがって、ラダーによる横・方向応答と同時に、速度、迎角、ピッチ姿勢も変化し得る。

また、`delta_e=None` かつ `trim_elevator=True` の場合、初期エレベータ舵角は、入力後のラダー舵角 $\delta_r=-10\ \mathrm{deg}$ を含めた初期ピッチングモーメント係数がゼロになるように計算される。`theta_hold=False` なので、このエレベータ舵角は積分中に固定される。

このため、本例は「ラダーだけを変化させ、エレベータを入力前の値に固定する純粋な単独ステップ」とは異なる。


In [ ]:
history = simulate_6dof_rudder_step_from_stab(
    stab_path,
    mass=mass,
    Ixx=inertia["Ixx"],
    Iyy=inertia["Iyy"],
    Izz=inertia["Izz"],
    Ixz=inertia["Ixz"],
    delta_r=delta_r,
    t_final=t_final,
    delta_a=0.0,
    delta_e=None,
    trim_elevator=True,
    trim_thrust=False,
    rho=rho,
    phi0=0.0,
    theta0=None,
    psi0=0.0,
    max_step=max_step,
)

display(history.head())
print("samples:", len(history))
print("initial elevator [deg]:", math.degrees(history.attrs["delta_e0"]))


## 7. 6 自由度有限時間ロール応答指標

指定した符号付きバンク角変化へ最初に到達する時刻を使う。

$$
K_{\phi,\delta_r}
=
\frac{b}{2V}
\frac{
\overline{\dot\phi}_{\Delta\phi}
}{
\delta_r
}
=
\frac{b}{2V}
\frac{
\Delta\phi/\Delta t
}{
\delta_r
}
$$

ここで $\Delta\phi/\Delta t$ は平均バンク角速度であり、機体軸ロール角速度 $p$ の平均ではない。

`V` と `Bref` には、線形モデルと同じ `.stab` から読み取った `Vinf` と `Bref` を渡す。


In [ ]:
sixdof_response = calculate_roll_response_index_by_delta_phi(
    history,
    delta_r=delta_r,
    target_delta_phi=target_delta_phi,
    V=stab.V0,
    Bref=stab.Bref,
    phi0=0.0,
)

display(
    pd.Series(
        {
            "reached": sixdof_response[
                "sixdof_roll_response_reached"
            ],
            "target_delta_phi_deg": sixdof_response[
                "sixdof_target_delta_phi_deg"
            ],
            "t_reach": sixdof_response["sixdof_t_reach"],
            "mean_bank_angle_rate": sixdof_response[
                "sixdof_roll_response_phi_rate"
            ],
            "phi_final_deg": math.degrees(
                sixdof_response["sixdof_phi_final"]
            ),
            "finite_time_roll_index": sixdof_response[
                "sixdof_finite_time_roll_index"
            ],
            "reference_index_to_t_final": sixdof_response[
                "sixdof_roll_response_index_reference"
            ],
            "error": sixdof_response[
                "sixdof_roll_response_error"
            ],
        },
        name="6DOF response",
    )
)


## 8. 線形モデルと 6 自由度モデルの比較

両モデルは同じ `.stab`、質量、$I_{xx}$、$I_{zz}$、$I_{xz}$、ラダー入力、目標バンク角を使用する。ただし、線形モデルは横・方向 4 状態の微小擾乱モデルであり、6 自由度モデルには縦運動、速度変化、姿勢運動が含まれる。


In [ ]:
comparison = pd.DataFrame(
    {
        "linear": {
            "reached": linear_response[
                "linear_roll_response_reached"
            ],
            "t_reach": linear_response["linear_t_reach"],
            "finite_time_roll_index": linear_response[
                "linear_finite_time_roll_index"
            ],
            "phi_final_deg": math.degrees(
                linear_response["linear_phi_final"]
            ),
        },
        "sixdof": {
            "reached": sixdof_response[
                "sixdof_roll_response_reached"
            ],
            "t_reach": sixdof_response["sixdof_t_reach"],
            "finite_time_roll_index": sixdof_response[
                "sixdof_finite_time_roll_index"
            ],
            "phi_final_deg": math.degrees(
                sixdof_response["sixdof_phi_final"]
            ),
        },
    }
)

display(comparison)


## 9. 6 自由度時刻歴の保存と表示


In [ ]:
csv_path = write_6dof_history_csv(
    history,
    output_dir / "rudder_step_6dof_history.csv",
)

plot_path = output_dir / "rudder_step_6dof_history.png"
plot_6dof_history(
    history,
    plot_path=plot_path,
    show=True,
    degrees=True,
)

print("csv_path:", csv_path)
print("plot_path:", plot_path)


## 10. 確認事項

- `linear_roll_response_reached` と `sixdof_roll_response_reached` を確認する。
- `linear_roll_response_error` と `sixdof_roll_response_error` が空文字であることを確認する。
- `simple_rudder_roll_index` と `simple_turn_rate` が一致することを確認する。
- Taylor 係数と簡易経路係数は異なる近似レベルの量であるため、同じ値になることを要求しない。
- `V`、`Bref`、`rho` が同じ `.stab` の値であることを確認する。
- ラダー舵角と目標バンク角の符号が意図どおりであることを確認する。
- 質量・慣性モーメントが対象機の単位系と一致することを確認する。
- 6 自由度初期状態は完全な定常滑空トリムではないことを踏まえて、速度、迎角、ピッチ姿勢の変化も確認する。
- 目標へ到達しない場合は、目標角、入力舵角、積分時間を物理的根拠に基づいて見直す。
